# Mission 04: Logging Middleware - 해답 노트북

이 노트북은 네 번째 미션의 완성된 솔루션 코드와 설명입니다.

In [1]:
# 1. 필요한 라이브러리 및 환경 로드
import sys
import os
from dotenv import load_dotenv

while not os.path.exists("app") and os.getcwd() != "/":
    os.chdir("..")
# 루트 폴더 기준의 경로 등록
sys.path.append(os.path.abspath("src"))
sys.path.append(os.path.abspath("."))  # sys.path.append("app") 대신 "." 등록이 파이썬 패키지 경로 탐색에 안전합니다.
load_dotenv(override=True)

from app.utils.llm import get_llm
from app.utils.context import AgentContext
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from app.tools import web_search

/home/hukim/env_langchain_123/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/home/hukim/env_langchain_123/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### [미션 1] LoggingMiddleware 구현하기

아래 코드는 `before_agent`, `after_agent`, `wrap_tool_call`을 구현하여 대화 세션 ID별로 로그 파일을 생성해 적재하는 솔루션 코드입니다.

In [2]:
import time
import json
import os
from typing import Any, Dict
from langchain.agents.middleware import AgentMiddleware

# 🌟 유틸리티 모듈에서 정규화 및 정화 함수 임포트
from app.utils.message_utils import normalize_content, sanitize_text

class StudentLoggingMiddleware(AgentMiddleware):
    def __init__(self, log_dir="./artifacts/logs"):
        self.log_dir = log_dir
        os.makedirs(self.log_dir, exist_ok=True)

    def _get_session_id(self, runtime, request=None) -> str:
        """컨텍스트 스레드에서 최우선의 session_id(thread_id)를 찾아냅니다."""
        try:
            from langchain_core.runnables.config import get_config_from_context
            config = get_config_from_context()
            if config:
                tid = config.get("configurable", {}).get("thread_id")
                if tid:
                    return tid
        except Exception:
            pass

        if request and hasattr(request, "runtime") and hasattr(request.runtime, "config") and request.runtime.config:
            tid = request.runtime.config.get("configurable", {}).get("thread_id")
            if tid:
                return tid

        if runtime and hasattr(runtime, "context"):
            tid = getattr(runtime.context, "session_id", None)
            if tid:
                return tid

        return "unknown"

    def before_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        """에이전트 전체 실행의 시작 로깅"""
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None

        start_time = time.time()
        
        # 🌟 사용자 질문도 normalize_content를 거치게 해 다중 구조 입력에 대응합니다.
        user_query = normalize_content(state.get("messages", [])[-1].content) if state.get("messages") else "unknown"
        session_id = self._get_session_id(runtime)
        
        if runtime and runtime.context:
            runtime.context.start_time = start_time
            runtime.context.user_query = user_query
            runtime.context.session_id = session_id

        print(f"\n🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===")
        print(f"📥 사용자 질문: {user_query}")
        return None

    def after_agent(self, state: Dict[str, Any], runtime: Any) -> Dict[str, Any] | None:
        """에이전트 전체 실행의 완료 및 감사 로그 생성"""
        logging_enabled = getattr(runtime.context, "logging_enabled", False) if runtime and runtime.context else False
        if not logging_enabled:
            return None

        start_time = getattr(runtime.context, "start_time", None) if runtime and runtime.context else None
        user_query = getattr(runtime.context, "user_query", "unknown") if runtime and runtime.context else "unknown"
        session_id = self._get_session_id(runtime)
        
        duration_ms = int((time.time() - start_time) * 1000) if start_time else 0
        print(f"📤 에이전트 실행 완료 (소요: {duration_ms}ms)")

        agent_response = ""
        dialogue_history = []
        messages = state.get("messages", [])
        if messages:
            # 🌟 최종 답변에서 Gemini thought_signature 등을 완전 제거하고 순수 텍스트만 정규화 추출합니다.
            agent_response = normalize_content(messages[-1].content)
            for msg in messages:
                dialogue_history.append({
                    "role": msg.type,
                    # 🌟 대화 전체 이력의 내용도 깨끗하게 정규화하고 정화합니다.
                    "content": sanitize_text(normalize_content(msg.content))
                })

        audit_log = {
            "event": "agent_execution",
            "session_id": session_id,
            "query": user_query,
            "response": agent_response,
            "dialogue_history": dialogue_history,
            "latency_ms": duration_ms,
            "status": "SUCCESS" if messages else "FAILED",
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        self._append_log(session_id, audit_log)
        return None

    def wrap_tool_call(self, request, handler):
        """개별 도구(Tool Call) 격발의 시작과 완료를 감싸서 로깅"""
        logging_enabled = getattr(request.runtime.context, "logging_enabled", False) if request.runtime and request.runtime.context else False
        if not logging_enabled:
            return handler(request)

        tool_name = request.tool_call.get("name", "unknown_tool")
        tool_args = request.tool_call.get("args", {})
        start_time = time.time()
        
        print(f"🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: {tool_name}({tool_args})")
        response = handler(request)
        
        duration_ms = int((time.time() - start_time) * 1000)
        print(f"🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: {tool_name} (소요: {duration_ms}ms)")
        
        session_id = self._get_session_id(request.runtime, request=request)
        if request.runtime and request.runtime.context:
            request.runtime.context.session_id = session_id
            
        tool_log = {
            "event": "tool_execution",
            "session_id": session_id,
            "tool_name": tool_name,
            "arguments": tool_args,
            "result": str(response),
            "latency_ms": duration_ms,
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        }
        self._append_log(session_id, tool_log)
        return response

    def _append_log(self, session_id, log_data):
        log_file = os.path.join(self.log_dir, f"{session_id}.jsonl")
        with open(log_file, "a", encoding="utf-8") as f:
            f.write(json.dumps(log_data, ensure_ascii=False) + "\n")


### [미션 2] 미들웨어가 연동된 에이전트 생성 및 검증

작성한 미들웨어를 create_agent에 등록하고 대화를 요청하여 로그 파일이 분리되어 저장되는지 검증합니다.

In [3]:
log_directory = "./artifacts/logs"
thread_id = "session_logging_test_04"
log_filepath = os.path.join(log_directory, f"{thread_id}.jsonl")

if os.path.exists(log_filepath):
    os.remove(log_filepath)

logging_middleware = StudentLoggingMiddleware(log_dir=log_directory)
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)

from app.prompts import CHATBOT_SYSTEM_PROMPT

agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=CHATBOT_SYSTEM_PROMPT,
    middleware=[logging_middleware],
    context_schema=AgentContext
)

context_obj = AgentContext(logging_enabled=True)

res = agent.invoke(
    {"messages": [HumanMessage(content="마케팅 기법 중 '바이럴 마케팅'에 대해 web_search 도구를 한 번 써서 간단히 검색하고 핵심 1문장으로 요약해줘.")]},
    config={"configurable": {"thread_id": thread_id}},
    context=context_obj
)
print("최종 답변:", res["messages"][-1].content)


🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===
📥 사용자 질문: 마케팅 기법 중 '바이럴 마케팅'에 대해 web_search 도구를 한 번 써서 간단히 검색하고 핵심 1문장으로 요약해줘.
🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '바이럴 마케팅 정의'})


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 264ms)
📤 에이전트 실행 완료 (소요: 4670ms)
최종 답변: [{'type': 'text', 'text': '바이럴 마케팅은 소비자들이 자발적으로 기업의 제품이나 서비스 정보를 입소문이나 온라인 콘텐츠를 통해 바이러스처럼 빠르게 전파하도록 유도하는 마케팅 전략입니다.', 'thought_signature': 'AY89a1+RRXk6l83rQeY+1lxpd9ZiwNzg3mamC7wULnupUPh1kZgnUvnRja/MkhGGrfgNvC3Nibc4y7PUTbOFSLZGXzfd8y0KYRlbQEoO94v1wYLJN7dgj6YzjV/nul/LyADfRlgO6P+7u/w3jpaYetz4G/hfxrl50VPAW/BCqcIVUVTww+krYeV3jyOmpIp8Ha6T2FkzHNoKOZYxFJTVrravVEabQvpPVHLbmquOdcC/fyZU7c3fjjZsgAXeG4FSC1tXP+Bq/pc9VpyBvxq33MfZWcvIigxBhQSX6KQF/UhGHBgndZ2mo/zhGpeVlAjeS4exZHY0xbzrfnc3iA+LlYa596qode8tjXGUnNZV+nlil9nSWsAMZ2P87u2ARycDvWF90mHSY4+ZO1a910pocaQrdeCNMSNwpRHfsWfRCH0JGKnvbtAMKLeFnj5zwzNA5Xq2tRZ7CDvPIZKsZVA0KZw4gDZRBgAgIRgnwaXdw0tFL1A27MmNHdKuHlGTpnhwk/k='}]


In [4]:
res = agent.invoke(
    {"messages": [HumanMessage(content="이번에는 바이럴 마케팅 관련한 서적을 검색해서 추천해줘.")]},
    config={"configurable": {"thread_id": thread_id}},
    context=context_obj
)
print("최종 답변:", res["messages"][-1].content)


🪵 [LoggingMiddleware] === 에이전트 실행 시작 ===
📥 사용자 질문: 이번에는 바이럴 마케팅 관련한 서적을 검색해서 추천해줘.
🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '바이럴 마케팅 책 추천 도서'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 146ms)


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '바이럴 마케팅 관련 책 추천'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 137ms)


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': '입소문 마케팅 책 추천'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 168ms)


...

🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': 'best viral marketing books'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 147ms)


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔧 [LoggingMiddleware] ➡️ 도구 격발 시작: web_search({'query': 'Contagious Jonah Berger book'})
🔧 [LoggingMiddleware] ⬅️ 도구 격발 완료: web_search (소요: 179ms)


/mnt/c/Users/hyoun/Desktop/harness_agent/app/tools/common.py:371: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


📤 에이전트 실행 완료 (소요: 36443ms)
최종 답변: [{'type': 'text', 'text': '바이럴 마케팅(Viral Marketing)은 소비자가 자발적으로 메시지를 퍼뜨려 바이러스처럼 빠르게 확산되도록 만드는 마케팅 기법입니다. \n\n단순히 "운이 좋아서" 뜨는 것이 아니라, **사람들이 공유하고 싶게 만드는 과학적인 원리와 심리학적 법칙**이 존재합니다. 이를 체계적으로 배울 수 있는 **바이럴 마케팅 분야의 세계적인 명저와 실무 필독서 5권**을 추천해 드립니다.\n\n---\n\n### 1. 《컨테이저스 전략적 입소문》 (Contagious)\n* **저자:** 조나 버거 (와튼스쿨 마케팅학 교수)\n* **한 줄 요약:** 바이럴 마케팅의 \'교과서\'. 왜 어떤 것은 유행하고 어떤 것은 묻히는가에 대한 과학적 분석.\n* **핵심 내용 (STEPPS 법칙):**\n  저자는 수백만 개의 콘텐츠와 제품을 분석하여 입소문을 유도하는 6가지 법칙을 제시합니다.\n  * **S**ocial Currency (사회적 화폐): 사람들이 남들에게 멋져 보이기 위해 공유하는 정보.\n  * **T**riggers (계기): 일상 속에서 자연스럽게 우리 제품을 떠올리게 만드는 자극.\n  * **E**motion (감성): 마음을 움직이는 고각성 감정(경외감, 분노, 흥분 등)이 공유를 유도함.\n  * **P**ublic (대중성): 눈에 잘 띄어야 사람들이 따라 하기 쉬움.\n  * **P**ractical Value (실용적 가치): 타인에게 실질적인 도움이 되는 유용한 정보.\n  * **S**tories (이야기): 메시지를 담은 흥미진진한 스토리텔링.\n* **추천 대상:** 바이럴 마케팅의 체계적인 이론과 프레임워크를 가장 먼저 정립하고 싶은 분.\n\n---\n\n### 2. 《스틱!》 (Made to Stick)\n* **저자:** 칩 히스, 댄 히스\n* **한 줄 요약:** 사람들의 뇌리에 착 달라붙어 절대 잊히지 않는 메시지

### [미션 3] 생성된 감사 로그 검증

실제로 지정된 경로에 JSON 라인으로 로그들이 정상 저장되었는지 프린트해봅니다.

In [5]:
if os.path.exists(log_filepath):
    print(f"📝 적재된 로그 내용 ({log_filepath}):")
    print("-" * 80)
    with open(log_filepath, "r", encoding="utf-8") as f:
        for line in f:
            print(line.strip())
    print("-" * 80)
else:
    print("❌ 로그 파일이 생성되지 않았습니다.")

📝 적재된 로그 내용 (./artifacts/logs/session_logging_test_04.jsonl):
--------------------------------------------------------------------------------
{"event": "tool_execution", "session_id": "session_logging_test_04", "tool_name": "web_search", "arguments": {"query": "바이럴 마케팅 정의"}, "result": "content=\"[Web Search Results for '바이럴 마케팅 정의']\\n1. [바이럴 마케팅 - 나무위키]: Jul 14, 2026 · 바이럴 마케팅 (viral marketing)은 잠정적인 소비자들 사이에 소문이나 여론을 조장하여 상품에 대한 정보가 끊임없이 전파되도록 유도하는 마케팅 전략의 일종이라고 알려졌다. 영단어 …\\n2. [바이럴 뜻과 바이럴 마케팅: 사례, 장단점, 성공 전략 - Waveon]: Nov 28, 2023 · 바이럴 마케팅은 제품이나 서비스가 소비자들 사이에서 자발적으로 전파되어, 빠르게 확산되는 마케팅 전략중 하나 입니다. 무분별한 광고에 지친 소비자들에게 일반적인 광고 캠페인과는 달리, 사람들의 …\\n3. ['바이럴'이란? 뜻, 종류, 장단점, 특징 알아보기]: Jan 9, 2025 · 바이럴 콘텐츠: 사람들 사이에서 자발적으로 공유되며, 빠르게 확산되는 모든 형태의 콘텐츠를 말해요. 바이럴 마케팅: 콘텐츠를 통해 자연스럽게 제품이나 브랜드를 홍보하는 마케팅 전략이에요. 바이럴의 …\\n4. [바이럴마케팅의 정의. 장단점. 예시 정리]: Dec 12, 2024 · 여기서 Viral 이란 Virus에서 나온 말입니다. 즉 바이러스처럼 사람들 사이에 빠르게 확산되고 공유되도록 유도하는 마케팅 전략이 바이럴 마케팅 입니다. 이 전략의 목표는 자연스럽게 소비자들이 …\\n5. [바이럴 마케팅이란? 바이럴 마케팅